In [1]:
import sqlite3
import pandas as pd

# =====================================================================
# 1. Database Connection & Data Loading with Clean DateTime Parsing
# =====================================================================
conn = sqlite3.connect('supermarket.db')
df = pd.read_csv('SuperMarket Analysis.csv')

# Clean Date column to standard ISO format (YYYY-MM-DD)
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

# SENIOR FIX 1: Parse the 12-hour AM/PM time format in Pandas and extract 24-hour Hour 
# This prevents SQLite's strftime from failing on '1:08:00 PM' and returning 'Other'
df['Hour'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p').dt.hour

# Save clean DataFrame to SQL
df.to_sql('sales', conn, if_exists='replace', index=False)

# ---------------------------------------------------------------------
# 2. Products Table Creation (For JOIN exercises)
# ---------------------------------------------------------------------
products_data = {
    'product_line': ['Health and beauty', 'Electronic accessories',
                     'Home and lifestyle', 'Sports and travel',
                     'Food and beverages', 'Fashion accessories'],
    'category':     ['Personal care', 'Tech', 'Home',
                     'Lifestyle', 'FMCG', 'Fashion'],
    'margin_pct':   [0.35, 0.22, 0.28, 0.31, 0.18, 0.40]
}
products_df = pd.DataFrame(products_data)
products_df.to_sql('products', conn, if_exists='replace', index=False)


# =====================================================================
# Q1: Which branch (A/B/C) has the highest average transaction value?
# =====================================================================
# NOTE (SENIOR FIX 2): If you renamed branches in your previous steps 
# (e.g., A -> Alex, B -> Giza, C -> Cairo), the query will reflect those names.
q1_query = """
    SELECT Branch, 
           AVG(Sales) AS avg_transaction_value
    FROM sales
    GROUP BY Branch
    ORDER BY avg_transaction_value DESC
    LIMIT 1;
"""
q1_df = pd.read_sql(q1_query, conn)
print("=== Q1: Highest Average Transaction Value Branch ===")
print(q1_df.to_string(index=False))
print(f"**Insight:** Branch {q1_df['Branch'].values[0]} generates the highest average transaction value, indicating that customers at this location are making larger, higher-value purchases per visit.")
print("\n" + "="*80 + "\n")


# =====================================================================
# Q2: What time of day generates most revenue? (using CASE WHEN)
# =====================================================================
# SENIOR FIX 1 (RESOLVED): Querying the pre-calculated 'Hour' column to 
# ensure correct time buckets without failing on 12-hour AM/PM strings.
q2_query = """
    SELECT 
        CASE 
            WHEN Hour BETWEEN 10 AND 11 THEN 'Morning (10-12)'
            WHEN Hour BETWEEN 12 AND 16 THEN 'Afternoon (12-17)'
            WHEN Hour BETWEEN 17 AND 20 THEN 'Evening (17-20)'
            ELSE 'Other'
        END AS time_of_day,
        SUM(Sales) AS total_revenue
    FROM sales
    GROUP BY time_of_day
    ORDER BY total_revenue DESC;
"""
q2_df = pd.read_sql(q2_query, conn)
print("=== Q2: Revenue by Time of Day ===")
print(q2_df.to_string(index=False))
top_time = q2_df['time_of_day'].values[0]
print(f"**Insight:** The {top_time} period generates the highest overall revenue, suggesting we should focus key promotional campaigns and schedule optimal floor coverage during these hours.")
print("\n" + "="*80 + "\n")


# =====================================================================
# Q3: Which payment method do Members prefer vs Normal customers?
# =====================================================================
q3_query = """
    WITH customer_payment_rank AS (
        SELECT `Customer type`, 
               Payment, 
               COUNT(*) AS usage_count,
               RANK() OVER (PARTITION BY `Customer type` ORDER BY COUNT(*) DESC) as rank
        FROM sales
        GROUP BY `Customer type`, Payment
    )
    SELECT `Customer type`, 
           Payment AS preferred_payment, 
           usage_count
    FROM customer_payment_rank
    WHERE rank = 1;
"""
q3_df = pd.read_sql(q3_query, conn)
print("=== Q3: Payment Method Preference ===")
print(q3_df.to_string(index=False))
print(f"**Insight:** By understanding the preferred payment methods of Members versus Normal customers, we can streamline our checkout operations and design targeted promotional partnerships with payment providers.")
print("\n" + "="*80 + "\n")


# =====================================================================
# Q4: What is the month-over-month revenue trend?
# =====================================================================
q4_query = """
    SELECT strftime('%Y-%m', Date) AS month,
           SUM(Sales) AS current_month_revenue,
           LAG(SUM(Sales), 1) OVER (ORDER BY strftime('%Y-%m', Date)) AS previous_month_revenue,
           ROUND(
               ((SUM(Sales) - LAG(SUM(Sales), 1) OVER (ORDER BY strftime('%Y-%m', Date))) / 
               LAG(SUM(Sales), 1) OVER (ORDER BY strftime('%Y-%m', Date))) * 100, 2
           ) AS percentage_growth
    FROM sales
    WHERE Date IS NOT NULL AND Date != ''
    GROUP BY month
    ORDER BY month;
"""
q4_df = pd.read_sql(q4_query, conn)
print("=== Q4: Month-over-Month Revenue Trend ===")
print(q4_df.to_string(index=False))
print(f"**Insight:** Monitoring the Month-over-Month (MoM) revenue changes allows us to evaluate our seasonal trajectory and plan inventory levels more efficiently for upcoming high-demand periods.")
print("\n" + "="*80 + "\n")


# =====================================================================
# Q5: Which product line has the highest estimated profit?
# =====================================================================
q5_query = """
    SELECT `Product line`,
           SUM(Sales) AS total_sales,
           SUM(Sales * 0.15) AS estimated_profit
    FROM sales
    GROUP BY `Product line`
    ORDER BY estimated_profit DESC
    LIMIT 1;
"""
q5_df = pd.read_sql(q5_query, conn)
print("=== Q5: Product Line with Highest Estimated Profit ===")
print(q5_df.to_string(index=False))
print(f"**Insight:** `{q5_df['Product line'].values[0]}` is our most profitable product category, making it an excellent candidate for aggressive marketing and prime in-store shelf placement.")
print("\n" + "="*80 + "\n")

# Connection close
conn.close()

=== Q1: Highest Average Transaction Value Branch ===
Branch  avg_transaction_value
  Giza             337.099715
**Insight:** Branch Giza generates the highest average transaction value, indicating that customers at this location are making larger, higher-value purchases per visit.


=== Q2: Revenue by Time of Day ===
      time_of_day  total_revenue
Afternoon (12-17)    148023.3405
  Evening (17-20)    113144.5980
  Morning (10-12)     61798.8105
**Insight:** The Afternoon (12-17) period generates the highest overall revenue, suggesting we should focus key promotional campaigns and schedule optimal floor coverage during these hours.


=== Q3: Payment Method Preference ===
Customer type preferred_payment  usage_count
       Member              Cash          192
       Normal           Ewallet          159
**Insight:** By understanding the preferred payment methods of Members versus Normal customers, we can streamline our checkout operations and design targeted promotional partnerships 